In [1]:
# === Train 5-feature XGBoost, tune on VAL, and save TRAIN predictions (wide 64-grid format) ===
import os, numpy as np, pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

# -------- paths --------
BASE      = "/home/pratyush/Desktop/DS_Project"
FEAT_DIR  = f"{BASE}/data/features"
OUT_CSV   = f"{BASE}/results/predicted_train_5feat_xgb.csv"  # <-- single output file
STEM      = "hog_color_glcm_lbp_pos"  # 5 features

# -------- helpers --------
def load_split_df(split):
    """Return the raw features dataframe for a split (keeps image/cell_id for pivot)."""
    path = os.path.join(FEAT_DIR, f"{split}_{STEM}_features.csv")
    df = pd.read_csv(path)
    return df

def Xy_from_df(df):
    X = df.drop(columns=["image","cell_id","label"]).values.astype(np.float32)
    y = df["label"].values.astype(int)
    return X, y

# -------- load splits --------
df_train = load_split_df("train")
df_val   = load_split_df("val")

X_train, y_train = Xy_from_df(df_train)
X_val,   y_val   = Xy_from_df(df_val)

# -------- model (same good params you used) --------
xgb = XGBClassifier(
    n_estimators=600,
    learning_rate=0.04,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.75,
    reg_lambda=1.0,
    min_child_weight=3,
    gamma=0.2,
    scale_pos_weight=2,
    objective='binary:logistic',
    eval_metric='logloss',
    n_jobs=-1,
    random_state=42
)

print("🚀 Training 5-feature XGBoost…")
xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

# -------- tune threshold on validation (maximize F1) --------
p_val = xgb.predict_proba(X_val)[:,1]
best_thr, best_f1 = 0.5, -1.0
for thr in np.linspace(0.30, 0.60, 31):
    f1 = f1_score(y_val, (p_val >= thr).astype(int))
    if f1 > best_f1:
        best_f1, best_thr = f1, thr
print(f"🎯 Best threshold on VAL = {best_thr:.2f} (F1={best_f1:.3f})")

# -------- predict on TRAIN and write wide CSV --------
p_train = xgb.predict_proba(X_train)[:,1]
df_train_pred = df_train.copy()
df_train_pred["pred"] = (p_train >= best_thr).astype(int)

# make sure cells are ordered c01..c64 per image
df_train_pred["cell_index"] = df_train_pred["cell_id"].str.extract(r"c(\d{2})").astype(int) - 1
df_train_pred = df_train_pred.sort_values(["image","cell_index"])

# pivot to wide: image | c01 … c64
cols64 = [f"c{i:02d}" for i in range(1,65)]
wide = df_train_pred.pivot_table(index="image", columns="cell_id", values="pred", aggfunc="first")
# ensure correct column order and type
wide = wide.reindex(columns=cols64)
wide = wide.fillna(0).astype(int).reset_index()

# save single CSV
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
wide.to_csv(OUT_CSV, index=False)
print("💾 Saved:", OUT_CSV)
print("Shape:", wide.shape)  # (num_train_images, 65)


🚀 Training 5-feature XGBoost…
🎯 Best threshold on VAL = 0.48 (F1=0.772)
💾 Saved: /home/pratyush/Desktop/DS_Project/results/predicted_train_5feat_xgb.csv
Shape: (301, 65)
